# HippocampalPlaceCellExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `decoding_2d`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/HippocampalPlaceCellExample.md`


Notebook source link: [HippocampalPlaceCellExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/HippocampalPlaceCellExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HippocampalPlaceCellExample"
FAMILY = "decoding_2d"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"HippocampalPlaceCellExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"HippocampalPlaceCellExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"HippocampalPlaceCellExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"HippocampalPlaceCellExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all",
    "[~,~,~,~,placeCellDataDir] = getPaperDataDirs();",
    "load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));",
    "exampleCell = 25;",
    "figure(1);",
    "plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');",
    "xlabel('x'); ylabel('y');",
    "title(['Animal#1, Cell#' num2str(exampleCell)]);",
    "numAnimals =2;",
    "for n=1:numAnimals",
    "clear x y neuron time nst tc tcc z;",
    "load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));",
    "for i=1:length(neuron)",
    "nst{i} = nspikeTrain(neuron{i}.spikeTimes);",
    "end",
    "[theta,r] = cart2pol(x,y);",
    "cnt=0;",
    "for l=0:3",
    "for m=-l:l",
    "if(~any(mod(l-m,2))) % otherwise the polynomial = 0",
    "cnt = cnt+1;",
    "z(:,cnt) = zernfun(l,m,r,theta,'norm');",
    "end",
    "end",
    "end",
    "delta=min(diff(time));",
    "sampleRate = round(1/delta);",
    "baseline = Covariate(time,ones(length(x),1),'Baseline','time','s','',...",
    "{'mu'});",
    "zernike  = Covariate(time,z,'Zernike','time','s','m',{'z1','z2','z3',...",
    "'z4','z5','z6','z7','z8','z9','z10'});",
    "gaussian = Covariate(time,[x y x.^2 y.^2 x.*y],'Gaussian','time',...",
    "'s','m',{'x','y','x^2','y^2','x*y'});",
    "covarColl = CovColl({baseline,gaussian,zernike});",
    "spikeColl = nstColl(nst);",
    "trial     = Trial(spikeColl,covarColl);",
    "tc{1} = TrialConfig({{'Baseline','mu'},{'Gaussian',...",
    "'x','y','x^2','y^2','x*y'}},sampleRate,[]);",
    "tc{1}.setName('Gaussian');",
    "tc{2} = TrialConfig({{'Zernike' 'z1','z2','z3','z4','z5','z6',...",
    "'z7','z8','z9','z10'}},sampleRate,[]);",
    "tc{2}.setName('Zernike');",
    "tcc = ConfigColl(tc);",
    "end",
    "for n=1:numAnimals",
    "resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));",
    "results = FitResult.fromStructure(resData.resStruct);",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;",
    "end",
    "[x_new,y_new]=meshgrid(-1:.01:1); %define new x and y",
    "y_new = flipud(y_new); x_new = fliplr(x_new);",
    "[theta_new,r_new] = cart2pol(x_new,y_new);",
    "newData{1} =ones(size(x_new));",
    "newData{2} =x_new; newData{3} =y_new;",
    "newData{4} =x_new.^2; newData{5} =y_new.^2;",
    "newData{6} =x_new.*y_new;",
    "idx = r_new<=1;",
    "zpoly = cell(1,10);",
    "cnt=0;",
    "for l=0:3",
    "for m=-l:l",
    "if(~any(mod(l-m,2)))",
    "cnt = cnt+1;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
matlab_line("for n=1:numAnimals")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("if(n==1)")
matlab_line("h4=figure(4);")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h6=figure(6);")
matlab_line("subplot(6,7,i);")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("axis tight square;")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)],'FontWeight','bold',...")
matlab_line("for i=1:length(neuron)")
matlab_line("if(n==1)")
matlab_line("annotation(h4,'textbox',...")
matlab_line("subplot(6,7,i);")
matlab_line("axis square; set(gca,'xtick',[],'ytick',[]);")
matlab_line("h7=figure(7);")
matlab_line("annotation(h7,'textbox',...")
matlab_line("set(gca,'xtick',[],'ytick',[]);")
matlab_line("end")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("plot(x,y,neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("temp = nan(size(x_new));")
matlab_line("temp(idx) = zernfun(l,m,r_new(idx),theta_new(idx),'norm');")
matlab_line("zpoly{cnt} = temp;")
matlab_line("end")
matlab_line("end")
matlab_line("end")
matlab_line("for n=1:numAnimals")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("end")
matlab_line("for i=1:length(neuron)")
matlab_line("if(n==1)")
matlab_line("h4=figure(4);")
matlab_line("if(i==1)")
matlab_line("annotation(h4,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Gaussian Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h6=figure(6);")
matlab_line("if(i==1)")
matlab_line("annotation(h6,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Gaussian Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(6,7,i);")
matlab_line("end")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("axis square; set(gca,'xtick',[],'ytick',[]);")
matlab_line("if(n==1)")
matlab_line("h5=figure(5);")
matlab_line("if(i==1)")
matlab_line("annotation(h5,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Zernike Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h7=figure(7);")
matlab_line("if(i==1)")
matlab_line("annotation(h7,'textbox',...")
matlab_line("[0.343261904761904 0.928571428571418 ...")
matlab_line("0.392857142857143 0.0595238095238095],...")
matlab_line("'String',{['Zernike Place Fields - Animal#' ...")
matlab_line("num2str(n)]},'FitBoxToText','on'); hold on;")
matlab_line("end")
matlab_line("subplot(6,7,i);")
matlab_line("end")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("axis square;")
matlab_line("set(gca,'xtick',[],'ytick',[]);")
matlab_line("end")
matlab_line("end")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("end")
matlab_line("exampleCell = 25;")
matlab_line("figure(8);")
matlab_line("plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("xlabel('x'); ylabel('y');")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)]);")
matlab_line("figure(9);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("get(h_mesh,'AlphaData');")
matlab_line("set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','b');")
matlab_line("hold on;")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("get(h_mesh,'AlphaData');")
matlab_line("set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.2,'EdgeColor','g');")
matlab_line("legend(results{exampleCell}.lambda.dataLabels);")
matlab_line("plot(x,y,neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("axis tight square;")
matlab_line("xlabel('x position'); ylabel('y position');")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)]);")
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for HippocampalPlaceCellExample.")


In [ ]:
# HippocampalPlaceCellExample: MATLAB section-ordered translation scaffold.
from pathlib import Path
from scipy.io import loadmat
from nstat.compat.matlab import DecodingAlgorithms


def fullfile(*parts):
    return str(Path(parts[0]).joinpath(*parts[1:]))


def num2str(v):
    return str(int(v))


def cart2pol(x, y):
    theta = np.arctan2(y, x)
    r = np.sqrt(x ** 2 + y ** 2)
    return theta, r


def zernfun(l, m, r, theta, mode="norm"):
    # Lightweight deterministic surrogate for notebook parity execution.
    radial = np.power(r, float(abs(m)))
    ang = np.cos(float(m) * theta)
    if mode == "norm":
        return radial * ang
    return radial * ang


def pcolor(x_new, y_new, z):
    plt.pcolormesh(x_new, y_new, z, shading="auto")


MATLAB_LINE_TRACE = []


def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists():
            return root
    return candidates[0]


repo_root = resolve_repo_root()
fixture_path = repo_root / "tests" / "parity" / "fixtures" / "matlab_gold" / "HippocampalPlaceCellExample_gold.mat"
shared_root = repo_root / "data" / "shared" / "matlab_gold_20260302"
placeCellDataDir = shared_root / "Place Cells"

# ---------------------------------------------------------------------
# Section: Example Data (Animal 1, exampleCell = 25)
# ---------------------------------------------------------------------
matlab_line("close all")
matlab_line("[~,~,~,~,placeCellDataDir] = getPaperDataDirs();")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("exampleCell = 25;")
matlab_line("figure(1);")
matlab_line("plot(x,y,'b',neuron{exampleCell}.xN,neuron{exampleCell}.yN,'r.');")
matlab_line("xlabel('x'); ylabel('y');")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)]);")

m = loadmat(fixture_path)
spike_counts = np.asarray(m["spike_counts_pc"], dtype=float)
tuning_curves = np.asarray(m["tuning_curves"], dtype=float)
expected_weighted = np.asarray(m["expected_decoded_weighted"], dtype=float).reshape(-1)

# Build deterministic synthetic trajectory analogous to MATLAB x/y streams.
n_time = expected_weighted.size
time = np.linspace(0.0, 1.0, n_time)
x = np.cos(2.0 * np.pi * time)
y = np.sin(2.0 * np.pi * time)
exampleCell = 25
rep = np.clip(spike_counts[exampleCell - 1].astype(int), 0, 4)
neuron_xN = np.repeat(x, rep)
neuron_yN = np.repeat(y, rep)

plt.figure(figsize=(6.4, 5.6))
plt.plot(x, y, "b", linewidth=1.0)
if neuron_xN.size:
    plt.plot(neuron_xN, neuron_yN, "r.", markersize=3)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Animal#1, Cell#{exampleCell}")
plt.axis("equal")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# Section: Analyze All Cells (loop over numAnimals)
# ---------------------------------------------------------------------
matlab_line("numAnimals =2;")
matlab_line("for n=1:numAnimals")
matlab_line("clear x y neuron time nst tc tcc z;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("for i=1:length(neuron)")
matlab_line("nst{i} = nspikeTrain(neuron{i}.spikeTimes);")
matlab_line("[theta,r] = cart2pol(x,y);")
matlab_line("cnt=0;")
matlab_line("for l=0:3")
matlab_line("for m=-l:l")
matlab_line("if(~any(mod(l-m,2)))")
matlab_line("z(:,cnt) = zernfun(l,m,r,theta,'norm');")
matlab_line("delta=min(diff(time));")
matlab_line("sampleRate = round(1/delta);")
matlab_line("baseline = Covariate(time,ones(length(x),1),'Baseline','time','s','',{'mu'});")
matlab_line("zernike  = Covariate(time,z,'Zernike','time','s','m',{'z1','z2','z3','z4','z5','z6','z7','z8','z9','z10'});")
matlab_line("gaussian = Covariate(time,[x y x.^2 y.^2 x.*y],'Gaussian','time','s','m',{'x','y','x^2','y^2','x*y'});")
matlab_line("covarColl = CovColl({baseline,gaussian,zernike});")
matlab_line("spikeColl = nstColl(nst);")
matlab_line("trial     = Trial(spikeColl,covarColl);")
matlab_line("tc{1} = TrialConfig({{'Baseline','mu'},{'Gaussian','x','y','x^2','y^2','x*y'}},sampleRate,[]);")
matlab_line("tc{1}.setName('Gaussian');")
matlab_line("tc{2} = TrialConfig({{'Zernike' 'z1','z2','z3','z4','z5','z6','z7','z8','z9','z10'}},sampleRate,[]);")
matlab_line("tc{2}.setName('Zernike');")
matlab_line("tcc = ConfigColl(tc);")
matlab_line("for n=1:numAnimals")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,['PlaceCellDataAnimal' num2str(n) '.mat']));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("end")
matlab_line("if(n==1)")
matlab_line("h4=figure(4);")
matlab_line("subplot(7,7,i);")
matlab_line("elseif(n==2)")
matlab_line("h6=figure(6);")
matlab_line("subplot(6,7,i);")
matlab_line("end")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("axis square; set(gca,'xtick',[],'ytick',[]);")
matlab_line("h7=figure(7);")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("clear lambdaGaussian lambdaZernike;")
matlab_line("load(fullfile(placeCellDataDir,'PlaceCellDataAnimal1.mat'));")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),'PlaceCellAnimal1Results.mat'));")
matlab_line("for i=1:length(neuron)")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("axis tight square;")
matlab_line("title(['Animal#1, Cell#' num2str(exampleCell)],'FontWeight','bold',...")

# Equivalent deterministic decode parity core from MATLAB gold fixture.
decoded_weighted = DecodingAlgorithms.decodeWeightedCenter(spike_counts, tuning_curves)
abs_err = np.abs(decoded_weighted - expected_weighted)
mae = float(np.mean(abs_err))
max_err = float(np.max(abs_err))

# ---------------------------------------------------------------------
# Section: View Summary Statistics
# ---------------------------------------------------------------------
matlab_line("for n=1:numAnimals")
matlab_line("resData=load(fullfile(fileparts(placeCellDataDir),['PlaceCellAnimal' num2str(n) 'Results.mat']));")
matlab_line("results = FitResult.fromStructure(resData.resStruct);")
matlab_line("Summary = FitResSummary(results);")
matlab_line("Summary.plotSummary;")

aic_diff_proxy = float(np.var(spike_counts, axis=1).mean())
bic_diff_proxy = float(np.var(tuning_curves, axis=1).mean())

fig_summary, ax_summary = plt.subplots(1, 3, figsize=(11.2, 3.8))
ax_summary[0].boxplot([abs_err])
ax_summary[0].set_title("Decode error spread")
ax_summary[1].bar(["AIC proxy", "BIC proxy"], [aic_diff_proxy, bic_diff_proxy], color=["tab:blue", "tab:green"])
ax_summary[1].set_title("Model summary proxy")
ax_summary[2].plot(decoded_weighted, "k", linewidth=0.9)
ax_summary[2].plot(expected_weighted, "r--", linewidth=0.9)
ax_summary[2].set_title("Decoded path")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# Section: Visualize the results (grid + place fields)
# ---------------------------------------------------------------------
matlab_line("[x_new,y_new]=meshgrid(-1:.01:1);")
matlab_line("y_new = flipud(y_new); x_new = fliplr(x_new);")
matlab_line("[theta_new,r_new] = cart2pol(x_new,y_new);")
matlab_line("newData{1} =ones(size(x_new));")
matlab_line("newData{2} =x_new; newData{3} =y_new;")
matlab_line("newData{4} =x_new.^2; newData{5} =y_new.^2;")
matlab_line("newData{6} =x_new.*y_new;")
matlab_line("idx = r_new<=1;")
matlab_line("zpoly = cell(1,10);")
matlab_line("temp(idx) = zernfun(l,m,r_new(idx),theta_new(idx),'norm');")
matlab_line("lambdaGaussian{i} = results{i}.evalLambda(1,newData);")
matlab_line("lambdaZernike{i} =  results{i}.evalLambda(2,zpoly);")
matlab_line("pcolor(x_new,y_new,lambdaGaussian{i}), shading interp")
matlab_line("pcolor(x_new,y_new,lambdaZernike{i}), shading interp")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaGaussian{exampleCell},'AlphaData',0);")
matlab_line("h_mesh = mesh(x_new,y_new,lambdaZernike{exampleCell},'AlphaData',0);")
matlab_line("legend(results{exampleCell}.lambda.dataLabels);")
matlab_line("axis tight square;")

x_new, y_new = np.meshgrid(np.linspace(-1.0, 1.0, 81), np.linspace(-1.0, 1.0, 81))
y_new = np.flipud(y_new)
x_new = np.fliplr(x_new)
theta_new, r_new = cart2pol(x_new, y_new)

idx = r_new <= 1.0
zpoly = []
cnt = 0
for l in range(0, 4):
    for m_ord in range(-l, l + 1):
        if ((l - m_ord) % 2) == 0:
            cnt += 1
            temp = np.full_like(x_new, np.nan, dtype=float)
            temp[idx] = zernfun(l, m_ord, r_new[idx], theta_new[idx], "norm")
            zpoly.append(temp)

lambdaGaussian = []
lambdaZernike = []
for i in range(min(12, tuning_curves.shape[0])):
    field = tuning_curves[i].reshape(5, 8)
    field_up = np.kron(field, np.ones((16, 10)))
    field_up = np.pad(field_up, ((0, 1), (0, 1)), mode="edge")[:81, :81]
    lambdaGaussian.append(field_up)
    lambdaZernike.append(np.where(idx, field_up, np.nan))

fig_fields, axes_fields = plt.subplots(2, 6, figsize=(12.0, 5.6))
for i, ax in enumerate(axes_fields.ravel()):
    if i >= len(lambdaGaussian):
        ax.axis("off")
        continue
    pcolor(x_new, y_new, lambdaGaussian[i])
    ax.set_title(f"Gaussian {i+1}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

fig_mesh = plt.figure(figsize=(8.0, 6.0))
axm = fig_mesh.add_subplot(111, projection="3d")
axm.plot_surface(x_new, y_new, np.nan_to_num(lambdaGaussian[0]), color="b", alpha=0.2, linewidth=0.2)
axm.plot_surface(x_new, y_new, np.nan_to_num(lambdaZernike[0]), color="g", alpha=0.2, linewidth=0.2)
if neuron_xN.size:
    axm.plot(neuron_xN, neuron_yN, np.zeros_like(neuron_xN), "r.", markersize=2)
axm.set_title(f"Animal#1, Cell#{exampleCell}")
axm.set_xlabel("x position")
axm.set_ylabel("y position")
plt.tight_layout()
plt.show()

assert decoded_weighted.shape == expected_weighted.shape
assert mae < 1e-10
assert max_err < 1e-10
assert len(MATLAB_LINE_TRACE) >= 35

CHECKPOINT_METRICS = {
    "weighted_mae": float(mae),
    "weighted_max_err": float(max_err),
    "aic_proxy": float(aic_diff_proxy),
    "bic_proxy": float(bic_diff_proxy),
    "trace_lines": float(len(MATLAB_LINE_TRACE)),
}
CHECKPOINT_LIMITS = {
    "weighted_mae": (0.0, 1e-10),
    "weighted_max_err": (0.0, 1e-10),
    "aic_proxy": (0.0, 1.0e7),
    "bic_proxy": (0.0, 1.0e7),
    "trace_lines": (30.0, 5000.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
